# Feature Engineering

Этот ноутбук позволяет проводить эксперименты по генерации признаков. Можно менять параметры отбора, добавлять или исключать блоки данных.

In [ ]:
import sys
import os
import pandas as pd
import logging
import matplotlib.pyplot as plt
import seaborn as sns

# Путь к src
sys.path.append(os.path.abspath('..'))

from src.features.pipeline import FeatureEngineeringPipeline

# Настройки
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)


## 1. Конфигурация эксперимента

In [ ]:
EXP_PARAMS = {
    "top_n_features": 250,        # Сколько признаков оставить после Ridge
    "use_cache": True,            # Использовать ли предобработанные .parquet
    "ridge_alpha": 1.0            # Регуляризация в Ridge
}

## 2. Запуск пайплайна с кастомными параметрами

In [ ]:
class ExperimentalPipeline(FeatureEngineeringPipeline):
    """Подкласс для быстрой модификации поведения пайплайна без правки исходников"""
    def _select_features_ridge(self, df, top_n=None):
        # Значение из EXP_PARAMS
        target_n = EXP_PARAMS["top_n_features"]
        return super()._select_features_ridge(df, top_n=target_n)

pipeline = ExperimentalPipeline(
    use_cache=EXP_PARAMS["use_cache"],
)

# Запуск FE
final_df = pipeline.run()

## 3. Анализ важности признаков
Посмотрим какие признаки оказались самыми полезными

In [ ]:
logger.info("Loading feature importance analysis")
importance_df = pd.read_csv('../data/processed/feature_importance.csv')
logger.info(f"Loaded {len(importance_df)} features")

logger.info("Building feature importance plot")
plt.figure(figsize=(10, 8))
sns.barplot(x="importance", y="feature", data=importance_df.head(20))
plt.title(f"Топ 20 признаков")
plt.tight_layout()
plt.show()

## 4. Проверка корреляций
Убедимся, что отобранные признаки не слишком сильно коррелируют между собой

In [ ]:
logger.info("Starting correlation analysis for top 15 features")
top_features = importance_df['feature'].head(15).tolist()
logger.info(f"Analyzing correlations for: {', '.join(top_features)}")

corr_matrix = final_df[top_features].corr()
logger.info("Correlation matrix computed")

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu', fmt=".2f", center=0)
plt.title("Матрица корреляций Топ-15 признаков")
plt.tight_layout()
plt.show()

Запускаем в терминале `mlflow ui --backend-store-uri sqlite:///mlflow.db`